# Hindsight Phase 3: Synthetic Paystub Image GeneratorGenerates 200 synthetic paystub PNG images with ground-truth JSONs for image-based CBA scoring.

In [ ]:
import os, json, random, math, io, hashlib
from datetime import date, timedelta
from collections import Counter, defaultdict
from pathlib import Path

# Install Pillow if needed (Colab)
try:
    from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "Pillow"])
    from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance

import numpy as np

# ── Output dirs ─────────────────────────────────────────────────────────────
BASE_DIR = Path("output")
IMG_DIR = BASE_DIR / "images"
GT_DIR = BASE_DIR / "ground_truth"
for d in (IMG_DIR, GT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Fonts ───────────────────────────────────────────────────────────────────
# Download Google Fonts if not cached
FONT_DIR = Path("fonts")
FONT_DIR.mkdir(exist_ok=True)

FONT_URLS = {
    "Roboto-Regular.ttf": "https://raw.githubusercontent.com/googlefonts/roboto-2/main/src/hinted/Roboto-Regular.ttf",
    "OpenSans-Variable.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/opensans/OpenSans%5Bwdth%2Cwght%5D.ttf",
    "CourierPrime-Regular.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/courierprime/CourierPrime-Regular.ttf",
}

# Also check for locally installed Roboto Slab
import shutil, urllib.request
local_roboto_slab = Path.home() / "Library" / "Fonts" / "RobotoSlab-Variable.ttf"
if local_roboto_slab.exists():
    dest = FONT_DIR / "RobotoSlab-Regular.ttf"
    if not dest.exists():
        shutil.copy2(local_roboto_slab, dest)
        print(f"Copied local Roboto Slab -> {dest}")

for fname, url in FONT_URLS.items():
    fpath = FONT_DIR / fname
    if not fpath.exists():
        try:
            print(f"Downloading {fname}...")
            urllib.request.urlretrieve(url, fpath)
        except Exception as e:
            print(f"  Warning: Could not download {fname}: {e}")

FONT_FILES = list(FONT_DIR.glob("*.ttf"))
if not FONT_FILES:
    # Fallback: use default font
    print("Warning: No TTF fonts found. Will use Pillow default font.")
print(f"Fonts ready: {[f.name for f in FONT_FILES]}")
print(f"Output dirs: {IMG_DIR}, {GT_DIR}")

In [ ]:
# ── 28 Canonical Concepts across 5 Families ────────────────────────────────
CANONICAL_CONCEPTS = {
    "compensation": [
        "regular_pay_current", "overtime_pay_current", "bonus_current",
        "gross_pay_current", "net_pay_current", "gross_pay_ytd",
        "net_pay_ytd", "total_deductions_current",
    ],
    "temporal": [
        "pay_date", "pay_period_start", "pay_period_end", "pay_frequency",
    ],
    "identity": [
        "employee_name", "employer_name", "employee_id", "ssn_last4",
    ],
    "deductions": [
        "federal_tax_withheld", "state_tax_withheld", "social_security_tax",
        "medicare_tax", "retirement_401k", "health_insurance",
        "total_deductions_ytd",
    ],
    "address": [
        "employee_address", "employee_city_state_zip",
        "employer_address", "employer_city_state_zip",
    ],
}

ALL_CONCEPTS = [c for fam in CANONICAL_CONCEPTS.values() for c in fam]
CONCEPT_TO_FAMILY = {}
for fam, members in CANONICAL_CONCEPTS.items():
    for m in members:
        CONCEPT_TO_FAMILY[m] = fam

LABEL_ALIASES = {
    "regular_pay_current":     ["Regular Pay", "Regular Earnings", "Base Pay", "Salary", "Regular Hours Pay", "Base Earnings"],
    "overtime_pay_current":    ["Overtime Pay", "OT Pay", "Overtime", "OT Earnings", "Overtime Earnings", "Extra Hours"],
    "bonus_current":           ["Bonus", "Bonus Pay", "Incentive Pay", "Performance Bonus", "Cash Bonus"],
    "gross_pay_current":       ["Gross Pay", "Total Earnings", "Period Earnings", "Gross", "Gross Earnings", "Total Gross"],
    "net_pay_current":         ["Net Pay", "Take Home Pay", "Net Amount", "Amount Due", "Net Earnings", "Check Amount"],
    "gross_pay_ytd":           ["Gross YTD", "YTD Gross", "Year-to-Date Gross", "YTD Earnings", "Total YTD Gross"],
    "net_pay_ytd":             ["Net YTD", "YTD Net", "Year-to-Date Net", "YTD Net Pay", "Total YTD Net"],
    "total_deductions_current":["Total Deductions", "Deductions Total", "Total Withholdings", "Current Deductions", "Period Deductions"],
    "pay_date":                ["Pay Date", "Check Date", "Payment Date", "Date Paid", "Issue Date"],
    "pay_period_start":        ["Period Start", "Pay Period Begin", "Start Date", "Period Beginning", "From"],
    "pay_period_end":          ["Period End", "Pay Period End", "End Date", "Period Ending", "Through", "To"],
    "pay_frequency":           ["Pay Frequency", "Frequency", "Pay Schedule", "Pay Cycle", "Period Type"],
    "employee_name":           ["Employee Name", "Employee", "Name", "Worker Name", "Associate Name", "Team Member"],
    "employer_name":           ["Company", "Employer", "Company Name", "Organization", "Employer Name", "Business Name"],
    "employee_id":             ["Employee ID", "Emp ID", "Badge #", "Worker ID", "ID Number", "EE ID"],
    "ssn_last4":               ["SSN", "SSN Last 4", "Last 4 SSN", "SS#", "Social Security", "SSN (Last 4)"],
    "federal_tax_withheld":    ["Federal Tax", "Fed Tax", "Federal W/H", "Federal Income Tax", "FIT", "Fed Withholding"],
    "state_tax_withheld":      ["State Tax", "State W/H", "State Income Tax", "SIT", "State Withholding", "St Tax"],
    "social_security_tax":     ["Social Security", "FICA-SS", "SS Tax", "OASDI", "Social Sec Tax", "FICA Social Security"],
    "medicare_tax":            ["Medicare", "FICA-Med", "Medicare Tax", "Med Tax", "FICA Medicare", "Medicare W/H"],
    "retirement_401k":         ["401(k)", "401k", "Retirement", "401(k) Contrib", "Retirement Plan", "401k Deduction"],
    "health_insurance":        ["Health Insurance", "Medical", "Health Plan", "Medical Insurance", "Health", "Group Health"],
    "total_deductions_ytd":    ["YTD Deductions", "Deductions YTD", "Year-to-Date Deductions", "Total Deductions YTD", "YTD W/H"],
    "employee_address":        ["Address", "Street Address", "Home Address", "Mailing Address", "Street"],
    "employee_city_state_zip": ["City/State/Zip", "City, State ZIP", "Location", "City State Zip", "CSZ"],
    "employer_address":        ["Company Address", "Business Address", "Corp Address", "Office Address", "Employer Addr"],
    "employer_city_state_zip": ["Company City/State/Zip", "Business Location", "Corp City/State/Zip", "Office Location"],
}

print(f"{len(ALL_CONCEPTS)} canonical concepts across {len(CANONICAL_CONCEPTS)} families")
for fam, concepts in CANONICAL_CONCEPTS.items():
    print(f"  {fam} ({len(concepts)}): {concepts}")
print(f"\nLabel aliases: {sum(len(v) for v in LABEL_ALIASES.values())} total variants across {len(LABEL_ALIASES)} concepts")

In [ ]:
# ── Paystub Data Generator ──────────────────────────────────────────────────

FIRST_NAMES = [
    "James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda",
    "David", "Elizabeth", "William", "Barbara", "Richard", "Susan", "Joseph", "Jessica",
    "Thomas", "Sarah", "Christopher", "Karen", "Charles", "Lisa", "Daniel", "Nancy",
    "Matthew", "Betty", "Anthony", "Margaret", "Mark", "Sandra", "Donald", "Ashley",
    "Steven", "Dorothy", "Paul", "Kimberly", "Andrew", "Emily", "Joshua", "Donna",
    "Kenneth", "Michelle", "Kevin", "Carol", "Brian", "Amanda", "George", "Melissa",
    "Timothy", "Deborah",
]

LAST_NAMES = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis",
    "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson",
    "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson",
    "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson",
    "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen",
    "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera",
    "Campbell", "Mitchell", "Carter", "Roberts",
]

MIDDLE_INITIALS = list("ABCDEFGHJKLMNPRSTW")

COMPANY_NAMES = [
    "Apex Manufacturing Co.", "Summit Healthcare Group", "Pacific Coast Logistics",
    "Horizon Financial Services", "Greenfield Agriculture Inc.", "Metro Construction LLC",
    "Valley Tech Solutions", "Coastal Energy Corp.", "Pinnacle Staffing Agency",
    "Lakewood Education Group", "Atlas Industrial Supply", "Brightstar Retail Inc.",
    "Continental Freight Lines", "Diamond Foods Corporation", "Eastside Medical Center",
    "Frontier Telecommunications", "Golden State Packaging", "Heartland Insurance Co.",
    "Ironworks Engineering Ltd.", "Keystone Properties Group", "Liberty Manufacturing",
    "Magnolia Hospitality Inc.", "Northgate Distribution Co.", "Oakridge Consulting",
    "Premier Auto Services", "Quantum Data Systems", "Redwood Forest Products",
    "Silverline Transport LLC", "Trident Security Services", "United Waste Management",
]

STREETS = [
    "123 Main Street", "456 Oak Avenue", "789 Elm Boulevard", "321 Pine Road",
    "654 Maple Drive", "987 Cedar Lane", "147 Birch Court", "258 Walnut Way",
    "369 Spruce Terrace", "741 Ash Street", "852 Willow Place", "963 Poplar Ave",
    "111 Industrial Parkway", "222 Commerce Blvd", "333 Technology Drive",
    "444 Business Circle", "555 Corporate Way", "666 Enterprise Blvd",
    "777 Market Street", "888 Harbor Drive",
]

CITIES_STATES_ZIPS = [
    "Los Angeles, CA 90001", "New York, NY 10001", "Chicago, IL 60601",
    "Houston, TX 77001", "Phoenix, AZ 85001", "Philadelphia, PA 19101",
    "San Antonio, TX 78201", "San Diego, CA 92101", "Dallas, TX 75201",
    "San Jose, CA 95101", "Austin, TX 73301", "Jacksonville, FL 32099",
    "Fort Worth, TX 76101", "Columbus, OH 43085", "Charlotte, NC 28201",
    "Indianapolis, IN 46201", "San Francisco, CA 94101", "Seattle, WA 98101",
    "Denver, CO 80201", "Nashville, TN 37201", "Portland, OR 97201",
    "Oklahoma City, OK 73101", "Las Vegas, NV 89101", "Memphis, TN 38101",
    "Louisville, KY 40201", "Baltimore, MD 21201", "Milwaukee, WI 53201",
    "Albuquerque, NM 87101", "Tucson, AZ 85701", "Fresno, CA 93701",
    "Mesa, AZ 85201", "Sacramento, CA 95801", "Atlanta, GA 30301",
    "Omaha, NE 68101", "Raleigh, NC 27601", "Miami, FL 33101",
    "Cleveland, OH 44101", "Tulsa, OK 74101", "Tampa, FL 33601",
    "Arlington, TX 76001",
]

PAY_FREQUENCIES = ["Weekly", "Bi-Weekly", "Semi-Monthly", "Monthly"]
FREQ_PERIODS_PER_YEAR = {"Weekly": 52, "Bi-Weekly": 26, "Semi-Monthly": 24, "Monthly": 12}
FREQ_PERIOD_DAYS = {"Weekly": 7, "Bi-Weekly": 14, "Semi-Monthly": 15, "Monthly": 30}


class PaystubDataGenerator:
    """Generate internally-consistent paystub data dictionaries."""

    def __init__(self, seed=None):
        self.rng = random.Random(seed)

    def generate(self) -> dict:
        """Return a dict with display-formatted string values + raw numeric ground truth."""
        rng = self.rng

        # ── Identity ────────────────────────────────────────────────────
        first = rng.choice(FIRST_NAMES)
        last = rng.choice(LAST_NAMES)
        mid = rng.choice(MIDDLE_INITIALS)
        employee_name = f"{first} {mid}. {last}"
        employer_name = rng.choice(COMPANY_NAMES)
        employee_id = str(rng.randint(100000, 999999))
        ssn_last4 = f"***-**-{rng.randint(1000, 9999)}"

        # ── Addresses ───────────────────────────────────────────────────
        emp_street = rng.choice(STREETS)
        emp_csz = rng.choice(CITIES_STATES_ZIPS)
        company_street = rng.choice(STREETS)
        company_csz = rng.choice(CITIES_STATES_ZIPS)

        # ── Temporal ────────────────────────────────────────────────────
        frequency = rng.choice(PAY_FREQUENCIES)
        periods_per_year = FREQ_PERIODS_PER_YEAR[frequency]
        period_days = FREQ_PERIOD_DAYS[frequency]

        # Pick a random period in 2024
        period_num = rng.randint(1, periods_per_year)
        year = 2024
        if frequency == "Monthly":
            period_start = date(year, period_num, 1)
            if period_num == 12:
                period_end = date(year, 12, 31)
            else:
                period_end = date(year, period_num + 1, 1) - timedelta(days=1)
        elif frequency == "Semi-Monthly":
            month = (period_num - 1) // 2 + 1
            if period_num % 2 == 1:  # 1st-15th
                period_start = date(year, month, 1)
                period_end = date(year, month, 15)
            else:  # 16th-end
                period_start = date(year, month, 16)
                if month == 12:
                    period_end = date(year, 12, 31)
                else:
                    period_end = date(year, month + 1, 1) - timedelta(days=1)
        else:
            day_offset = (period_num - 1) * period_days
            period_start = date(year, 1, 1) + timedelta(days=day_offset)
            period_end = period_start + timedelta(days=period_days - 1)
            if period_end.year > year:
                period_end = date(year, 12, 31)

        pay_date = period_end + timedelta(days=rng.randint(3, 7))

        # ── Compensation ────────────────────────────────────────────────
        annual_salary = rng.uniform(30000, 120000)
        regular_pay = round(annual_salary / periods_per_year, 2)

        has_overtime = rng.random() < 0.60
        overtime_pay = round(rng.uniform(100, 800), 2) if has_overtime else 0.0

        has_bonus = rng.random() < 0.40
        bonus = round(rng.uniform(200, 2000), 2) if has_bonus else 0.0

        gross_current = round(regular_pay + overtime_pay + bonus, 2)

        # ── Deductions ──────────────────────────────────────────────────
        federal_tax = round(gross_current * rng.uniform(0.10, 0.22), 2)
        state_tax = round(gross_current * rng.uniform(0.03, 0.09), 2)
        ss_tax = round(gross_current * 0.062, 2)
        medicare_tax = round(gross_current * 0.0145, 2)

        has_401k = rng.random() < 0.70
        retirement = round(gross_current * rng.uniform(0.03, 0.10), 2) if has_401k else 0.0

        has_health = rng.random() < 0.75
        health_ins = round(rng.uniform(80, 350), 2) if has_health else 0.0

        total_ded_current = round(federal_tax + state_tax + ss_tax + medicare_tax + retirement + health_ins, 2)
        net_current = round(gross_current - total_ded_current, 2)

        # ── YTD ─────────────────────────────────────────────────────────
        gross_ytd = round(gross_current * period_num, 2)
        net_ytd = round(net_current * period_num, 2)
        ded_ytd = round(total_ded_current * period_num, 2)

        # ── Format helpers ──────────────────────────────────────────────
        def fmt_money(v):
            return f"${v:,.2f}"

        # ── Build data dict (display values) ────────────────────────────
        data = {
            "employee_name": employee_name,
            "employer_name": employer_name,
            "employee_id": employee_id,
            "ssn_last4": ssn_last4,
            "employee_address": emp_street,
            "employee_city_state_zip": emp_csz,
            "employer_address": company_street,
            "employer_city_state_zip": company_csz,
            "pay_date": pay_date.strftime("%m/%d/%Y"),
            "pay_period_start": period_start.strftime("%m/%d/%Y"),
            "pay_period_end": period_end.strftime("%m/%d/%Y"),
            "pay_frequency": frequency,
            "regular_pay_current": fmt_money(regular_pay),
            "gross_pay_current": fmt_money(gross_current),
            "net_pay_current": fmt_money(net_current),
            "gross_pay_ytd": fmt_money(gross_ytd),
            "net_pay_ytd": fmt_money(net_ytd),
            "total_deductions_current": fmt_money(total_ded_current),
            "federal_tax_withheld": fmt_money(federal_tax),
            "state_tax_withheld": fmt_money(state_tax),
            "social_security_tax": fmt_money(ss_tax),
            "medicare_tax": fmt_money(medicare_tax),
            "total_deductions_ytd": fmt_money(ded_ytd),
        }

        # Optional fields
        if has_overtime:
            data["overtime_pay_current"] = fmt_money(overtime_pay)
        if has_bonus:
            data["bonus_current"] = fmt_money(bonus)
        if has_401k:
            data["retirement_401k"] = fmt_money(retirement)
        if has_health:
            data["health_insurance"] = fmt_money(health_ins)

        # ── Ground truth (raw numeric strings for scoring) ──────────────
        ground_truth = {
            "employee_name": employee_name,
            "employer_name": employer_name,
            "employee_id": employee_id,
            "ssn_last4": ssn_last4,
            "employee_address": emp_street,
            "employee_city_state_zip": emp_csz,
            "employer_address": company_street,
            "employer_city_state_zip": company_csz,
            "pay_date": pay_date.strftime("%m/%d/%Y"),
            "pay_period_start": period_start.strftime("%m/%d/%Y"),
            "pay_period_end": period_end.strftime("%m/%d/%Y"),
            "pay_frequency": frequency,
            "regular_pay_current": f"{regular_pay:.2f}",
            "gross_pay_current": f"{gross_current:.2f}",
            "net_pay_current": f"{net_current:.2f}",
            "gross_pay_ytd": f"{gross_ytd:.2f}",
            "net_pay_ytd": f"{net_ytd:.2f}",
            "total_deductions_current": f"{total_ded_current:.2f}",
            "federal_tax_withheld": f"{federal_tax:.2f}",
            "state_tax_withheld": f"{state_tax:.2f}",
            "social_security_tax": f"{ss_tax:.2f}",
            "medicare_tax": f"{medicare_tax:.2f}",
            "total_deductions_ytd": f"{ded_ytd:.2f}",
        }
        if has_overtime:
            ground_truth["overtime_pay_current"] = f"{overtime_pay:.2f}"
        if has_bonus:
            ground_truth["bonus_current"] = f"{bonus:.2f}"
        if has_401k:
            ground_truth["retirement_401k"] = f"{retirement:.2f}"
        if has_health:
            ground_truth["health_insurance"] = f"{health_ins:.2f}"

        return {"display": data, "ground_truth": ground_truth}

    def validate(self, gt: dict) -> bool:
        """Check internal math consistency."""
        def g(k):
            return float(gt.get(k, "0"))

        regular = g("regular_pay_current")
        overtime = g("overtime_pay_current")
        bonus = g("bonus_current")
        gross = g("gross_pay_current")
        expected_gross = round(regular + overtime + bonus, 2)
        if abs(gross - expected_gross) > 0.02:
            return False

        fed = g("federal_tax_withheld")
        state = g("state_tax_withheld")
        ss = g("social_security_tax")
        med = g("medicare_tax")
        ret = g("retirement_401k")
        health = g("health_insurance")
        total_ded = g("total_deductions_current")
        expected_ded = round(fed + state + ss + med + ret + health, 2)
        if abs(total_ded - expected_ded) > 0.02:
            return False

        net = g("net_pay_current")
        expected_net = round(gross - total_ded, 2)
        if abs(net - expected_net) > 0.02:
            return False

        return True


# Quick test
gen = PaystubDataGenerator(seed=42)
sample = gen.generate()
assert gen.validate(sample["ground_truth"]), "Validation failed!"
print("Sample paystub data:")
for k, v in sorted(sample["display"].items()):
    print(f"  {k}: {v}")
print(f"\nValidation: PASSED")

In [ ]:
# ── Layout Templates (15 distinct layouts) ─────────────────────────────────

def draw_field(draw, concept_id, x, y, data, labels, font_label, font_value, record_bbox, label_above=True, spacing=16):
    """Draw a label+value pair and record bbox. Returns y after drawing."""
    if concept_id not in data:
        return y
    label_text = labels.get(concept_id, concept_id)
    value_text = data[concept_id]
    if label_above:
        draw.text((x, y), label_text, fill=(100, 100, 100), font=font_label)
        y += spacing
        bbox = font_value.getbbox(value_text)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        draw.text((x, y), value_text, fill=(0, 0, 0), font=font_value)
        record_bbox(concept_id, x, y, w, h)
        y += spacing + 4
    else:
        label_w = font_label.getbbox(label_text)[2] - font_label.getbbox(label_text)[0]
        draw.text((x, y), label_text, fill=(100, 100, 100), font=font_label)
        bbox = font_value.getbbox(value_text)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        draw.text((x + label_w + 10, y), value_text, fill=(0, 0, 0), font=font_value)
        record_bbox(concept_id, x + label_w + 10, y, w, h)
        y += spacing + 4
    return y


def draw_table_row(draw, label_text, value_text, concept_id, x, y, col_width, font_label, font_value, record_bbox, fill=(0, 0, 0)):
    """Draw a table row with label on left, value right-aligned."""
    draw.text((x, y), label_text, fill=(100, 100, 100), font=font_label)
    if value_text:
        bbox = font_value.getbbox(value_text)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        vx = x + col_width - w
        draw.text((vx, y), value_text, fill=fill, font=font_value)
        if concept_id:
            record_bbox(concept_id, vx, y, w, h)
    return y + 20


def draw_separator(draw, x1, x2, y, color=(180, 180, 180)):
    draw.line([(x1, y), (x2, y)], fill=color, width=1)


# ── Helper lists used across layouts ───────────────────────────────────────
EARNINGS_IDS = ["regular_pay_current", "overtime_pay_current", "bonus_current",
                "gross_pay_current", "gross_pay_ytd"]
DEDUCTION_IDS = ["federal_tax_withheld", "state_tax_withheld", "social_security_tax",
                 "medicare_tax", "retirement_401k", "health_insurance",
                 "total_deductions_current", "total_deductions_ytd"]
IDENTITY_IDS = ["employee_name", "employee_id", "ssn_last4",
                "employee_address", "employee_city_state_zip"]
DATE_IDS = ["pay_date", "pay_period_start", "pay_period_end", "pay_frequency"]
EMPLOYER_ADDR_IDS = ["employer_address", "employer_city_state_zip"]


# ═══════════════════════════════════════════════════════════════════════════
# 1. corporate_standard — Centered header, left-aligned tables
# ═══════════════════════════════════════════════════════════════════════════
def draw_corporate_standard(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 40

    # Centered employer name
    if "employer_name" in data:
        txt = data["employer_name"]
        bb = fonts["header"].getbbox(txt)
        tw, th = bb[2]-bb[0], bb[3]-bb[1]
        ex = (img_width - tw) // 2
        draw.text((ex, y), txt, fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", ex, y, tw, th)
    y += 35

    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            tw, th = bb[2]-bb[0], bb[3]-bb[1]
            cx = (img_width - tw) // 2
            draw.text((cx, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, cx, y, tw, th)
            y += 14
    y += 10
    draw_separator(draw, margin, img_width-margin, y)
    y += 15

    for cid in IDENTITY_IDS:
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["value"], record_bbox)
    y += 10
    draw_separator(draw, margin, img_width-margin, y)
    y += 15

    # Two-column: earnings left, deductions right
    left_x, right_x, col_w = margin, 450, 350
    draw.text((left_x, y), "EARNINGS", fill=(0,0,0), font=fonts["label"])
    draw.text((right_x, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"])
    y += 22
    draw_separator(draw, left_x, left_x+col_w, y)
    draw_separator(draw, right_x, right_x+col_w, y)
    y += 5

    ey, dy = y, y
    for cid in EARNINGS_IDS:
        if cid in data:
            ey = draw_table_row(draw, labels.get(cid, cid), data[cid], cid, left_x, ey, col_w, fonts["label"], fonts["value"], record_bbox)
    for cid in DEDUCTION_IDS:
        if cid in data:
            dy = draw_table_row(draw, labels.get(cid, cid), data[cid], cid, right_x, dy, col_w, fonts["label"], fonts["value"], record_bbox)
    y = max(ey, dy) + 20
    draw_separator(draw, margin, img_width-margin, y)
    y += 15

    y = draw_field(draw, "net_pay_current", margin, y, data, labels, fonts["label"], fonts["header"], record_bbox)
    y = draw_field(draw, "net_pay_ytd", margin+300, y, data, labels, fonts["label"], fonts["value"], record_bbox)

    y += 20
    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            draw.text((dx, y), labels.get(cid, cid), fill=(100,100,100), font=fonts["small"])
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx, y+13), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, dx, y+13, bb[2]-bb[0], bb[3]-bb[1])
            dx += 190


# ═══════════════════════════════════════════════════════════════════════════
# 2. left_header — Left-aligned company block
# ═══════════════════════════════════════════════════════════════════════════
def draw_left_header(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 40

    # Dates top-right
    dx = img_width - 200
    dy = y
    for cid in DATE_IDS:
        if cid in data:
            lbl = labels.get(cid, cid) + ":"
            draw.text((dx, dy), lbl, fill=(100,100,100), font=fonts["small"])
            lw = fonts["small"].getbbox(lbl + " ")[2]
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx+lw, dy), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, dx+lw, dy, bb[2]-bb[0], bb[3]-bb[1])
            dy += 14

    # Employer top-left
    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
        y += 30
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y += 20
    draw_separator(draw, margin, img_width-margin, y)
    y += 15

    # Employee right-aligned block
    rx = img_width - 350
    ey = y
    for cid in IDENTITY_IDS:
        if cid in data:
            lbl = labels.get(cid, cid) + ":"
            draw.text((rx, ey), lbl, fill=(100,100,100), font=fonts["label"])
            lw = fonts["label"].getbbox(lbl + " ")[2]
            bb = fonts["value"].getbbox(data[cid])
            draw.text((rx+lw+5, ey), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, rx+lw+5, ey, bb[2]-bb[0], bb[3]-bb[1])
            ey += 20
    y = max(y, ey) + 15
    draw_separator(draw, margin, img_width-margin, y)
    y += 10

    full_w = img_width - 2*margin
    draw.text((margin, y), "EARNINGS", fill=(0,0,0), font=fonts["label"])
    y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in EARNINGS_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    y += 10
    draw_separator(draw, margin, img_width-margin, y); y += 10

    draw.text((margin, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"])
    y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in DEDUCTION_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    y += 15
    draw_separator(draw, margin, img_width-margin, y); y += 10

    for cid in ("net_pay_current", "net_pay_ytd"):
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["header"], record_bbox)


# ═══════════════════════════════════════════════════════════════════════════
# 3. two_column — Earnings left, deductions right
# ═══════════════════════════════════════════════════════════════════════════
def draw_two_column(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 40

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
        y += 30
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y += 8
    for cid in IDENTITY_IDS:
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["value"], record_bbox, label_above=False)

    y += 5
    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            draw_field(draw, cid, dx, y, data, labels, fonts["small"], fonts["small"], record_bbox, label_above=False)
            dx += 200
    y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 10

    left_x, right_x = margin, 450
    left_w, right_w = 350, 350
    draw.text((left_x, y), "EARNINGS", fill=(0,0,0), font=fonts["label"])
    draw.text((right_x, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"])
    y += 20
    draw_separator(draw, left_x, left_x+left_w, y)
    draw_separator(draw, right_x, right_x+right_w, y)
    y += 5
    ey, dy = y, y
    for cid in EARNINGS_IDS:
        if cid in data:
            ey = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, left_x, ey, left_w, fonts["label"], fonts["value"], record_bbox)
    for cid in DEDUCTION_IDS:
        if cid in data:
            dy = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, right_x, dy, right_w, fonts["label"], fonts["value"], record_bbox)
    y = max(ey, dy) + 15
    draw_separator(draw, margin, img_width-margin, y); y += 15

    draw.text((margin, y), "SUMMARY", fill=(0,0,0), font=fonts["label"])
    y += 22
    sx = margin
    for cid in ("gross_pay_current", "total_deductions_current", "net_pay_current"):
        if cid in data:
            draw.text((sx, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["label"])
            bb = fonts["header"].getbbox(data[cid])
            draw.text((sx, y+18), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, sx, y+18, bb[2]-bb[0], bb[3]-bb[1])
            sx += 260


# ═══════════════════════════════════════════════════════════════════════════
# 4. compact — Dense single-column, small font
# ═══════════════════════════════════════════════════════════════════════════
def draw_compact(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y, sp = 40, 25, 14
    fs, fv = fonts["small"], fonts["small"]

    def inline(cid, x, y):
        if cid not in data: return y
        lbl = labels.get(cid, cid) + ":"
        draw.text((x, y), lbl, fill=(100,100,100), font=fs)
        lw = fs.getbbox(lbl + " ")[2]
        bb = fv.getbbox(data[cid])
        draw.text((x+lw, y), data[cid], fill=(0,0,0), font=fv)
        record_bbox(cid, x+lw, y, bb[2]-bb[0], bb[3]-bb[1])
        return y + sp

    for cid in ("employer_name", "employee_name", "employee_id", "ssn_last4"):
        y = inline(cid, margin, y)
    for cid in EMPLOYER_ADDR_IDS + ["employee_address", "employee_city_state_zip"]:
        if cid in data:
            bb = fs.getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fs)
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += sp - 2
    y += 4; draw_separator(draw, margin, img_width-margin, y); y += 4
    for cid in DATE_IDS:
        y = inline(cid, margin, y)
    y += 4; draw_separator(draw, margin, img_width-margin, y); y += 4

    col_w = img_width - 2*margin
    draw.text((margin, y), "Earnings", fill=(0,0,0), font=fs); y += sp
    for cid in EARNINGS_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, col_w, fs, fs, record_bbox)
    y += 4; draw_separator(draw, margin, img_width-margin, y); y += 4

    draw.text((margin, y), "Deductions", fill=(0,0,0), font=fs); y += sp
    for cid in DEDUCTION_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, col_w, fs, fs, record_bbox)
    y += 4; draw_separator(draw, margin, img_width-margin, y); y += 4

    for cid in ("net_pay_current", "net_pay_ytd"):
        y = inline(cid, margin, y)


# ═══════════════════════════════════════════════════════════════════════════
# 5. check_style — Right-aligned numbers, check-like
# ═══════════════════════════════════════════════════════════════════════════
def draw_check_style(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 40

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
    if "employee_id" in data:
        txt = "Check #" + data["employee_id"]
        bb = fonts["value"].getbbox(txt)
        cx = img_width - margin - (bb[2]-bb[0])
        draw.text((cx, y+5), txt, fill=(0,0,0), font=fonts["value"])
        record_bbox("employee_id", cx, y+5, bb[2]-bb[0], bb[3]-bb[1])
    y += 45
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y += 10

    # Employee box
    bx, by, bw, bh = margin, y, 350, 90
    draw.rectangle([bx, by, bx+bw, by+bh], outline=(0,0,0), width=2)
    iy = by + 8
    for cid in ("employee_name", "ssn_last4", "employee_address", "employee_city_state_zip"):
        if cid in data:
            lbl = labels.get(cid,cid) + ":"
            draw.text((bx+10, iy), lbl, fill=(100,100,100), font=fonts["small"])
            lw = fonts["small"].getbbox(lbl + " ")[2]
            bb = fonts["value"].getbbox(data[cid])
            draw.text((bx+10+lw, iy), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, bx+10+lw, iy, bb[2]-bb[0], bb[3]-bb[1])
            iy += 18

    # Dates beside box
    ddx, ddy = bx+bw+30, by+8
    for cid in DATE_IDS:
        if cid in data:
            lbl = labels.get(cid,cid) + ":"
            draw.text((ddx, ddy), lbl, fill=(100,100,100), font=fonts["small"])
            lw = fonts["small"].getbbox(lbl + " ")[2]
            bb = fonts["small"].getbbox(data[cid])
            draw.text((ddx+lw, ddy), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, ddx+lw, ddy, bb[2]-bb[0], bb[3]-bb[1])
            ddy += 16

    y = by + bh + 20
    draw_separator(draw, margin, img_width-margin, y); y += 10

    left_x, right_x, col_w = margin, 450, 350
    draw.text((left_x, y), "EARNINGS", fill=(0,0,0), font=fonts["label"])
    draw.text((right_x, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"])
    y += 22
    draw_separator(draw, left_x, left_x+col_w, y)
    draw_separator(draw, right_x, right_x+col_w, y)
    y += 5
    ey, dy = y, y
    for cid in EARNINGS_IDS:
        if cid in data:
            ey = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, left_x, ey, col_w, fonts["label"], fonts["value"], record_bbox)
    for cid in DEDUCTION_IDS:
        if cid in data:
            dy = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, right_x, dy, col_w, fonts["label"], fonts["value"], record_bbox)
    y = max(ey, dy) + 20

    # Net pay box
    if "net_pay_current" in data:
        npw = img_width - 2*margin
        draw.rectangle([margin, y, margin+npw, y+60], outline=(0,0,0), width=3, fill=(240,248,255))
        draw.text((margin+15, y+8), labels.get("net_pay_current","Net Pay") + ":", fill=(100,100,100), font=fonts["label"])
        bb = fonts["header"].getbbox(data["net_pay_current"])
        draw.text((margin+15, y+28), data["net_pay_current"], fill=(0,0,0), font=fonts["header"])
        record_bbox("net_pay_current", margin+15, y+28, bb[2]-bb[0], bb[3]-bb[1])
        if "net_pay_ytd" in data:
            bb2 = fonts["value"].getbbox(data["net_pay_ytd"])
            w2 = bb2[2]-bb2[0]
            draw.text((margin+npw-w2-100, y+8), labels.get("net_pay_ytd","YTD") + ":", fill=(100,100,100), font=fonts["small"])
            draw.text((margin+npw-w2-15, y+30), data["net_pay_ytd"], fill=(0,0,0), font=fonts["value"])
            record_bbox("net_pay_ytd", margin+npw-w2-15, y+30, w2, bb2[3]-bb2[1])


# ═══════════════════════════════════════════════════════════════════════════
# 6. modern_clean — Minimal sans-serif, gray bands
# ═══════════════════════════════════════════════════════════════════════════
def draw_modern_clean(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 60, 50

    draw.rectangle([0, y-10, img_width, y+90], fill=(245,245,245))
    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(30,30,30), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
    y += 35
    ix = margin
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((ix, y), data[cid], fill=(100,100,100), font=fonts["small"])
            record_bbox(cid, ix, y, bb[2]-bb[0], bb[3]-bb[1])
            ix += (bb[2]-bb[0]) + 15
    y += 40

    ex = margin
    for cid in ("employee_name", "employee_id", "ssn_last4"):
        if cid in data:
            draw.text((ex, y), labels.get(cid,cid), fill=(150,150,150), font=fonts["small"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((ex, y+14), data[cid], fill=(30,30,30), font=fonts["value"])
            record_bbox(cid, ex, y+14, bb[2]-bb[0], bb[3]-bb[1])
            ex += 200
    y += 40

    for cid in ("employee_address", "employee_city_state_zip"):
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 15
    y += 15

    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            draw.text((dx, y), labels.get(cid,cid), fill=(150,150,150), font=fonts["small"])
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx, y+13), data[cid], fill=(50,50,50), font=fonts["small"])
            record_bbox(cid, dx, y+13, bb[2]-bb[0], bb[3]-bb[1])
            dx += 185
    y += 40

    left_x, right_x = margin, img_width//2+20
    col_w = img_width//2 - margin - 20
    draw.text((left_x, y), "Earnings", fill=(80,80,80), font=fonts["label"])
    draw.text((right_x, y), "Deductions", fill=(80,80,80), font=fonts["label"])
    y += 25
    ey, dy = y, y
    for cid in EARNINGS_IDS:
        if cid in data:
            ey = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, left_x, ey, col_w, fonts["label"], fonts["value"], record_bbox)
    for cid in DEDUCTION_IDS:
        if cid in data:
            dy = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, right_x, dy, col_w, fonts["label"], fonts["value"], record_bbox)
    y = max(ey, dy) + 30

    sx = margin
    for cid in ("net_pay_current", "net_pay_ytd"):
        if cid in data:
            draw.text((sx, y), labels.get(cid,cid), fill=(150,150,150), font=fonts["small"])
            bb = fonts["header"].getbbox(data[cid])
            draw.text((sx, y+15), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, sx, y+15, bb[2]-bb[0], bb[3]-bb[1])
            sx += 300


# ═══════════════════════════════════════════════════════════════════════════
# 7. government — Bold centered header, formal dense
# ═══════════════════════════════════════════════════════════════════════════
def draw_government(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 30
    title = "EARNINGS STATEMENT"
    bb = fonts["header"].getbbox(title)
    draw.text(((img_width-(bb[2]-bb[0]))//2, y), title, fill=(0,0,0), font=fonts["header"])
    y += 35
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=3)
    y += 10

    def dense_row(cid, x, y):
        if cid not in data: return y
        lbl = labels.get(cid,cid) + ": "
        draw.text((x, y), lbl, fill=(0,0,0), font=fonts["label"])
        lw = fonts["label"].getbbox(lbl)[2]
        bb = fonts["value"].getbbox(data[cid])
        draw.text((x+lw, y), data[cid], fill=(0,0,0), font=fonts["value"])
        record_bbox(cid, x+lw, y, bb[2]-bb[0], bb[3]-bb[1])
        return y + 20

    for cid in ("employer_name",) + tuple(EMPLOYER_ADDR_IDS):
        y = dense_row(cid, margin, y)
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=2); y += 8

    for cid in IDENTITY_IDS:
        y = dense_row(cid, margin, y)
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=2); y += 8

    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            lbl = labels.get(cid,cid) + ": "
            draw.text((dx, y), lbl, fill=(0,0,0), font=fonts["label"])
            lw = fonts["label"].getbbox(lbl)[2]
            bb = fonts["value"].getbbox(data[cid])
            draw.text((dx+lw, y), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, dx+lw, y, bb[2]-bb[0], bb[3]-bb[1])
            dx += 200
    y += 25
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=3); y += 8

    full_w = img_width - 2*margin
    draw.text((margin, y), "EARNINGS", fill=(0,0,0), font=fonts["label"]); y += 20
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=1); y += 5
    for cid in EARNINGS_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
            draw.line([(margin, y-2), (img_width-margin, y-2)], fill=(200,200,200), width=1)
    y += 5
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=2); y += 8

    draw.text((margin, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"]); y += 20
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=1); y += 5
    for cid in DEDUCTION_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
            draw.line([(margin, y-2), (img_width-margin, y-2)], fill=(200,200,200), width=1)
    y += 5
    draw.line([(margin, y), (img_width-margin, y)], fill=(0,0,0), width=3); y += 10

    for cid in ("net_pay_current", "net_pay_ytd"):
        if cid in data:
            lbl = labels.get(cid,cid) + ": "
            draw.text((margin, y), lbl, fill=(0,0,0), font=fonts["label"])
            lw = fonts["label"].getbbox(lbl)[2]
            bb = fonts["header"].getbbox(data[cid])
            draw.text((margin+lw, y), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, margin+lw, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 30


# ═══════════════════════════════════════════════════════════════════════════
# 8. detailed_stub — Full addresses, all fields visible
# ═══════════════════════════════════════════════════════════════════════════
def draw_detailed_stub(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 30

    draw.text((margin, y), "EMPLOYER", fill=(100,100,100), font=fonts["small"]); y += 14
    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
        y += 28
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["value"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(60,60,60), font=fonts["value"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 18
    y += 10; draw_separator(draw, margin, img_width-margin, y); y += 10

    draw.text((margin, y), "EMPLOYEE", fill=(100,100,100), font=fonts["small"]); y += 14
    if "employee_name" in data:
        bb = fonts["header"].getbbox(data["employee_name"])
        draw.text((margin, y), data["employee_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employee_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
        y += 28
    for cid in ("employee_id", "ssn_last4"):
        y = draw_field(draw, cid, margin, y, data, labels, fonts["small"], fonts["value"], record_bbox, label_above=False, spacing=16)
    for cid in ("employee_address", "employee_city_state_zip"):
        if cid in data:
            bb = fonts["value"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(60,60,60), font=fonts["value"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 18
    y += 10; draw_separator(draw, margin, img_width-margin, y); y += 10

    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            draw.text((dx, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["small"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((dx, y+14), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, dx, y+14, bb[2]-bb[0], bb[3]-bb[1])
            dx += 190
    y += 35; draw_separator(draw, margin, img_width-margin, y); y += 10

    full_w = img_width - 2*margin
    draw.text((margin, y), "EARNINGS DETAIL", fill=(0,0,0), font=fonts["label"]); y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in EARNINGS_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    draw_separator(draw, margin, img_width-margin, y+2); y += 12

    draw.text((margin, y), "DEDUCTIONS DETAIL", fill=(0,0,0), font=fonts["label"]); y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in DEDUCTION_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    draw_separator(draw, margin, img_width-margin, y+2); y += 15

    draw.text((margin, y), "NET PAY SUMMARY", fill=(0,0,0), font=fonts["label"]); y += 22
    for cid in ("net_pay_current", "net_pay_ytd"):
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["header"], record_bbox)


# ═══════════════════════════════════════════════════════════════════════════
# 9. hourly_detailed — Hours x Rate x Amount columns
# ═══════════════════════════════════════════════════════════════════════════
def draw_hourly_detailed(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 35

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
        y += 30
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y += 10; draw_separator(draw, margin, img_width-margin, y); y += 10

    for cid in ("employee_name", "employee_id", "ssn_last4"):
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["value"], record_bbox, label_above=False)
    for cid in ("employee_address", "employee_city_state_zip"):
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y += 10
    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            lbl = labels.get(cid,cid) + ":"
            draw.text((dx, y), lbl, fill=(100,100,100), font=fonts["small"])
            lw = fonts["small"].getbbox(lbl + " ")[2]
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx+lw, y), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, dx+lw, y, bb[2]-bb[0], bb[3]-bb[1])
            dx += 200
    y += 25; draw_separator(draw, margin, img_width-margin, y); y += 10

    # Hourly columns
    cd, ch, cr, ca = margin, margin+300, margin+450, margin+600
    ce = img_width - margin
    draw.text((cd, y), "Description", fill=(0,0,0), font=fonts["label"])
    draw.text((ch, y), "Hours", fill=(0,0,0), font=fonts["label"])
    draw.text((cr, y), "Rate", fill=(0,0,0), font=fonts["label"])
    draw.text((ca, y), "Amount", fill=(0,0,0), font=fonts["label"])
    y += 20; draw_separator(draw, margin, img_width-margin, y); y += 5

    for cid, hrs in [("regular_pay_current","80.00"), ("overtime_pay_current","10.00"), ("bonus_current","-")]:
        if cid in data:
            draw.text((cd, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["label"])
            draw.text((ch, y), hrs, fill=(100,100,100), font=fonts["value"])
            draw.text((cr, y), "-", fill=(100,100,100), font=fonts["value"])
            bb = fonts["value"].getbbox(data[cid])
            vx = ce - (bb[2]-bb[0])
            draw.text((vx, y), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, vx, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5

    full_w = img_width - 2*margin
    for cid in ("gross_pay_current", "gross_pay_ytd"):
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    y += 10; draw_separator(draw, margin, img_width-margin, y); y += 10

    draw.text((margin, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"]); y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in DEDUCTION_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    y += 15; draw_separator(draw, margin, img_width-margin, y); y += 10

    for cid in ("net_pay_current", "net_pay_ytd"):
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["header"], record_bbox)


# ═══════════════════════════════════════════════════════════════════════════
# 10. retail — Indented list, no table structure
# ═══════════════════════════════════════════════════════════════════════════
def draw_retail(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, indent, y = 60, 30, 40

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        tw = bb[2]-bb[0]
        cx = (img_width-tw)//2
        draw.text((cx, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", cx, y, tw, bb[3]-bb[1])
    y += 35
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            tw = bb[2]-bb[0]
            draw.text(((img_width-tw)//2, y), data[cid], fill=(100,100,100), font=fonts["small"])
            record_bbox(cid, (img_width-tw)//2, y, tw, bb[3]-bb[1])
            y += 14
    y += 20

    for cid in IDENTITY_IDS:
        if cid in data:
            draw.text((margin, y), labels.get(cid,cid), fill=(120,120,120), font=fonts["label"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((margin+indent, y+16), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, margin+indent, y+16, bb[2]-bb[0], bb[3]-bb[1])
            y += 34
    y += 10
    for cid in DATE_IDS:
        if cid in data:
            draw.text((margin, y), labels.get(cid,cid), fill=(120,120,120), font=fonts["label"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((margin+indent, y+16), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, margin+indent, y+16, bb[2]-bb[0], bb[3]-bb[1])
            y += 34
    y += 10

    draw.text((margin, y), "Earnings", fill=(60,60,60), font=fonts["label"]); y += 22
    for cid in EARNINGS_IDS:
        if cid in data:
            draw.text((margin+indent, y), labels.get(cid,cid), fill=(120,120,120), font=fonts["label"])
            bb = fonts["value"].getbbox(data[cid])
            vx = img_width - margin - (bb[2]-bb[0])
            draw.text((vx, y), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, vx, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 22
    y += 10

    draw.text((margin, y), "Deductions", fill=(60,60,60), font=fonts["label"]); y += 22
    for cid in DEDUCTION_IDS:
        if cid in data:
            draw.text((margin+indent, y), labels.get(cid,cid), fill=(120,120,120), font=fonts["label"])
            bb = fonts["value"].getbbox(data[cid])
            vx = img_width - margin - (bb[2]-bb[0])
            draw.text((vx, y), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, vx, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 22
    y += 15

    for cid in ("net_pay_current", "net_pay_ytd"):
        if cid in data:
            draw.text((margin, y), labels.get(cid,cid), fill=(60,60,60), font=fonts["label"])
            bb = fonts["header"].getbbox(data[cid])
            vx = img_width - margin - (bb[2]-bb[0])
            draw.text((vx, y), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, vx, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 30


# ═══════════════════════════════════════════════════════════════════════════
# 11. reversed_sections — Deductions ABOVE earnings
# ═══════════════════════════════════════════════════════════════════════════
def draw_reversed_sections(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 50, 35

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
        y += 30
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y += 10; draw_separator(draw, margin, img_width-margin, y); y += 10
    for cid in IDENTITY_IDS:
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["value"], record_bbox, label_above=False)
    y += 5
    dx = margin
    for cid in DATE_IDS:
        if cid in data:
            lbl = labels.get(cid,cid) + ":"
            draw.text((dx, y), lbl, fill=(100,100,100), font=fonts["small"])
            lw = fonts["small"].getbbox(lbl + " ")[2]
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx+lw, y), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, dx+lw, y, bb[2]-bb[0], bb[3]-bb[1])
            dx += 200
    y += 25; draw_separator(draw, margin, img_width-margin, y); y += 10

    full_w = img_width - 2*margin
    # DEDUCTIONS FIRST
    draw.text((margin, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"]); y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in DEDUCTION_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    y += 10; draw_separator(draw, margin, img_width-margin, y); y += 10

    # EARNINGS SECOND
    draw.text((margin, y), "EARNINGS", fill=(0,0,0), font=fonts["label"]); y += 20
    draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in EARNINGS_IDS:
        if cid in data:
            y = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin, y, full_w, fonts["label"], fonts["value"], record_bbox)
    y += 15; draw_separator(draw, margin, img_width-margin, y); y += 10
    for cid in ("net_pay_current", "net_pay_ytd"):
        y = draw_field(draw, cid, margin, y, data, labels, fonts["label"], fonts["header"], record_bbox)


# ═══════════════════════════════════════════════════════════════════════════
# 12. side_by_side — Identity left, money right
# ═══════════════════════════════════════════════════════════════════════════
def draw_side_by_side(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin = 50
    le, rs, re = 400, 430, img_width-margin
    yl = 40

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, yl), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, yl, bb[2]-bb[0], bb[3]-bb[1])
        yl += 30
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, yl), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, yl, bb[2]-bb[0], bb[3]-bb[1])
            yl += 14
    yl += 15; draw_separator(draw, margin, le, yl); yl += 10
    for cid in ("employee_name", "employee_id", "ssn_last4"):
        yl = draw_field(draw, cid, margin, yl, data, labels, fonts["label"], fonts["value"], record_bbox)
    for cid in ("employee_address", "employee_city_state_zip"):
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, yl), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin, yl, bb[2]-bb[0], bb[3]-bb[1])
            yl += 15
    yl += 10; draw_separator(draw, margin, le, yl); yl += 10
    for cid in DATE_IDS:
        yl = draw_field(draw, cid, margin, yl, data, labels, fonts["label"], fonts["value"], record_bbox)

    draw.line([(le+15, 40), (le+15, img_height-100)], fill=(180,180,180), width=1)

    yr = 40
    rcw = re - rs
    draw.text((rs, yr), "EARNINGS", fill=(0,0,0), font=fonts["label"]); yr += 20
    draw_separator(draw, rs, re, yr); yr += 5
    for cid in EARNINGS_IDS:
        if cid in data:
            yr = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, rs, yr, rcw, fonts["label"], fonts["value"], record_bbox)
    yr += 10; draw_separator(draw, rs, re, yr); yr += 10
    draw.text((rs, yr), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"]); yr += 20
    draw_separator(draw, rs, re, yr); yr += 5
    for cid in DEDUCTION_IDS:
        if cid in data:
            yr = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, rs, yr, rcw, fonts["label"], fonts["value"], record_bbox)
    yr += 15; draw_separator(draw, rs, re, yr); yr += 10
    draw.text((rs, yr), "SUMMARY", fill=(0,0,0), font=fonts["label"]); yr += 22
    for cid in ("net_pay_current", "net_pay_ytd"):
        if cid in data:
            draw.text((rs, yr), labels.get(cid,cid), fill=(100,100,100), font=fonts["small"])
            bb = fonts["header"].getbbox(data[cid])
            draw.text((rs, yr+14), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, rs, yr+14, bb[2]-bb[0], bb[3]-bb[1])
            yr += 45


# ═══════════════════════════════════════════════════════════════════════════
# 13. boxed — Heavy gridlines/borders
# ═══════════════════════════════════════════════════════════════════════════
def draw_boxed(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y, pad = 40, 30, 8
    bw = img_width - 2*margin

    def box(x, y, w, h, lbl=None):
        draw.rectangle([x, y, x+w, y+h], outline=(0,0,0), width=2)
        if lbl: draw.text((x+pad, y+pad), lbl, fill=(100,100,100), font=fonts["small"])

    # Employer box
    bh = 65
    box(margin, y, bw, bh, "EMPLOYER")
    iy = y + 22
    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin+pad, iy), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin+pad, iy, bb[2]-bb[0], bb[3]-bb[1])
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin+pad+300, iy), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, margin+pad+300, iy, bb[2]-bb[0], bb[3]-bb[1])
            iy += 14
    y += bh + 10

    # Employee box
    bh = 95
    box(margin, y, bw, bh, "EMPLOYEE")
    iy = y + 22
    for cid in IDENTITY_IDS:
        if cid in data:
            lbl = labels.get(cid,cid) + ":"
            draw.text((margin+pad, iy), lbl, fill=(100,100,100), font=fonts["small"])
            lw = fonts["small"].getbbox(lbl + " ")[2]
            bb = fonts["value"].getbbox(data[cid])
            draw.text((margin+pad+lw, iy), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, margin+pad+lw, iy, bb[2]-bb[0], bb[3]-bb[1])
            iy += 15
    y += bh + 10

    # Dates box
    bh = 40
    box(margin, y, bw, bh, "PAY PERIOD")
    dx = margin + pad + 100
    for cid in DATE_IDS:
        if cid in data:
            lbl = labels.get(cid,cid) + ":"
            draw.text((dx, y+pad), lbl, fill=(100,100,100), font=fonts["small"])
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx, y+pad+14), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, dx, y+pad+14, bb[2]-bb[0], bb[3]-bb[1])
            dx += 170
    y += bh + 10

    # Earnings + Deductions side-by-side boxes
    hw = (bw - 10) // 2
    e_cnt = sum(1 for c in EARNINGS_IDS if c in data)
    d_cnt = sum(1 for c in DEDUCTION_IDS if c in data)
    bh = 25 + max(e_cnt, d_cnt) * 20 + 10
    box(margin, y, hw, bh, "EARNINGS")
    iy = y + 22
    for cid in EARNINGS_IDS:
        if cid in data:
            iy = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, margin+pad, iy, hw-2*pad, fonts["small"], fonts["small"], record_bbox)
    dx2 = margin + hw + 10
    box(dx2, y, hw, bh, "DEDUCTIONS")
    iy = y + 22
    for cid in DEDUCTION_IDS:
        if cid in data:
            iy = draw_table_row(draw, labels.get(cid,cid), data[cid], cid, dx2+pad, iy, hw-2*pad, fonts["small"], fonts["small"], record_bbox)
    y += bh + 10

    # Net pay box
    bh = 55
    box(margin, y, bw, bh, "NET PAY")
    nx = margin + pad + 80
    for cid in ("net_pay_current", "net_pay_ytd"):
        if cid in data:
            draw.text((nx, y+pad), labels.get(cid,cid), fill=(100,100,100), font=fonts["small"])
            bb = fonts["header"].getbbox(data[cid])
            draw.text((nx, y+pad+16), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, nx, y+pad+16, bb[2]-bb[0], bb[3]-bb[1])
            nx += 300


# ═══════════════════════════════════════════════════════════════════════════
# 14. minimal — Almost no labels, implied structure
# ═══════════════════════════════════════════════════════════════════════════
def draw_minimal(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 70, 50

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
    y += 35

    if "employee_name" in data:
        bb = fonts["value"].getbbox(data["employee_name"])
        w = bb[2]-bb[0]
        draw.text((img_width-margin-w, y), data["employee_name"], fill=(0,0,0), font=fonts["value"])
        record_bbox("employee_name", img_width-margin-w, y, w, bb[3]-bb[1])

    ix = margin
    for cid in ("employee_id", "ssn_last4"):
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((ix, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, ix, y, bb[2]-bb[0], bb[3]-bb[1])
            ix += (bb[2]-bb[0]) + 30
    y += 25

    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((margin, y), data[cid], fill=(130,130,130), font=fonts["small"])
            record_bbox(cid, margin, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 13
    ea_x, ea_y = img_width//2, y-26
    for cid in ("employee_address", "employee_city_state_zip"):
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((ea_x, ea_y), data[cid], fill=(130,130,130), font=fonts["small"])
            record_bbox(cid, ea_x, ea_y, bb[2]-bb[0], bb[3]-bb[1])
            ea_y += 13
    y += 15

    dx = margin
    for cid in ("pay_date", "pay_period_start", "pay_period_end"):
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((dx, y), data[cid], fill=(0,0,0), font=fonts["small"])
            record_bbox(cid, dx, y, bb[2]-bb[0], bb[3]-bb[1])
            dx += (bb[2]-bb[0]) + 40
    if "pay_frequency" in data:
        bb = fonts["small"].getbbox(data["pay_frequency"])
        draw.text((dx, y), data["pay_frequency"], fill=(100,100,100), font=fonts["small"])
        record_bbox("pay_frequency", dx, y, bb[2]-bb[0], bb[3]-bb[1])
    y += 30
    draw_separator(draw, margin, img_width-margin, y, color=(220,220,220)); y += 15

    c1, c2, c3 = margin, margin+250, margin+500
    py = y
    for cid in ("regular_pay_current", "overtime_pay_current", "bonus_current"):
        if cid in data:
            short = labels.get(cid,cid).split()[0]
            draw.text((c1, py), short, fill=(160,160,160), font=fonts["small"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((c1+80, py), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, c1+80, py, bb[2]-bb[0], bb[3]-bb[1])
            py += 22
    py2 = y
    for cid in ("federal_tax_withheld", "state_tax_withheld", "social_security_tax", "medicare_tax"):
        if cid in data:
            bb = fonts["value"].getbbox(data[cid])
            draw.text((c2, py2), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, c2, py2, bb[2]-bb[0], bb[3]-bb[1])
            py2 += 22
    py3 = y
    for cid in ("retirement_401k", "health_insurance"):
        if cid in data:
            bb = fonts["value"].getbbox(data[cid])
            draw.text((c3, py3), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, c3, py3, bb[2]-bb[0], bb[3]-bb[1])
            py3 += 22
    y = max(py, py2, py3) + 20
    draw_separator(draw, margin, img_width-margin, y, color=(220,220,220)); y += 15

    tx = margin
    for cid in ("gross_pay_current", "total_deductions_current", "net_pay_current"):
        if cid in data:
            draw.text((tx, y), labels.get(cid,cid), fill=(130,130,130), font=fonts["small"])
            bb = fonts["header"].getbbox(data[cid])
            draw.text((tx, y+14), data[cid], fill=(0,0,0), font=fonts["header"])
            record_bbox(cid, tx, y+14, bb[2]-bb[0], bb[3]-bb[1])
            tx += 250
    y += 45
    tx = margin
    for cid in ("gross_pay_ytd", "total_deductions_ytd", "net_pay_ytd"):
        if cid in data:
            draw.text((tx, y), labels.get(cid,cid), fill=(160,160,160), font=fonts["small"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((tx, y+14), data[cid], fill=(60,60,60), font=fonts["value"])
            record_bbox(cid, tx, y+14, bb[2]-bb[0], bb[3]-bb[1])
            tx += 250


# ═══════════════════════════════════════════════════════════════════════════
# 15. wide_format — Landscape-ish, wide tables
# ═══════════════════════════════════════════════════════════════════════════
def draw_wide_format(draw, data, labels, fonts, record_bbox, img_width=850, img_height=1100):
    margin, y = 40, 30

    if "employer_name" in data:
        bb = fonts["header"].getbbox(data["employer_name"])
        draw.text((margin, y), data["employer_name"], fill=(0,0,0), font=fonts["header"])
        record_bbox("employer_name", margin, y, bb[2]-bb[0], bb[3]-bb[1])
    ea_x = 400
    for cid in EMPLOYER_ADDR_IDS:
        if cid in data:
            bb = fonts["small"].getbbox(data[cid])
            draw.text((ea_x, y), data[cid], fill=(80,80,80), font=fonts["small"])
            record_bbox(cid, ea_x, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 14
    y = max(y, 55) + 5
    draw_separator(draw, margin, img_width-margin, y); y += 8

    pairs = [("employee_name",margin), ("employee_id",margin+220), ("ssn_last4",margin+400), ("pay_date",margin+550)]
    for cid, xx in pairs:
        if cid in data:
            draw.text((xx, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["small"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((xx, y+13), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, xx, y+13, bb[2]-bb[0], bb[3]-bb[1])
    y += 32
    pairs2 = [("employee_address",margin), ("employee_city_state_zip",margin+220), ("pay_period_start",margin+450), ("pay_period_end",margin+600)]
    for cid, xx in pairs2:
        if cid in data:
            draw.text((xx, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["small"])
            bb = fonts["value"].getbbox(data[cid])
            draw.text((xx, y+13), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, xx, y+13, bb[2]-bb[0], bb[3]-bb[1])
    y += 32
    if "pay_frequency" in data:
        draw.text((margin, y), labels.get("pay_frequency","Frequency"), fill=(100,100,100), font=fonts["small"])
        bb = fonts["value"].getbbox(data["pay_frequency"])
        draw.text((margin, y+13), data["pay_frequency"], fill=(0,0,0), font=fonts["value"])
        record_bbox("pay_frequency", margin, y+13, bb[2]-bb[0], bb[3]-bb[1])
        y += 30
    draw_separator(draw, margin, img_width-margin, y); y += 10

    cc, cy = margin+400, margin+600
    draw.text((margin, y), "EARNINGS", fill=(0,0,0), font=fonts["label"])
    draw.text((cc, y), "Current", fill=(0,0,0), font=fonts["label"])
    draw.text((cy, y), "YTD", fill=(0,0,0), font=fonts["label"])
    y += 20; draw_separator(draw, margin, img_width-margin, y); y += 5

    for cid in ("regular_pay_current", "overtime_pay_current", "bonus_current"):
        if cid in data:
            draw.text((margin, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["label"])
            bb = fonts["value"].getbbox(data[cid])
            vx = cc + 150 - (bb[2]-bb[0])
            draw.text((vx, y), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, vx, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 20

    if "gross_pay_current" in data:
        draw_separator(draw, margin, img_width-margin, y); y += 5
        draw.text((margin, y), labels.get("gross_pay_current","Gross"), fill=(0,0,0), font=fonts["label"])
        bb = fonts["value"].getbbox(data["gross_pay_current"])
        vx = cc + 150 - (bb[2]-bb[0])
        draw.text((vx, y), data["gross_pay_current"], fill=(0,0,0), font=fonts["value"])
        record_bbox("gross_pay_current", vx, y, bb[2]-bb[0], bb[3]-bb[1])
        if "gross_pay_ytd" in data:
            bb2 = fonts["value"].getbbox(data["gross_pay_ytd"])
            vx2 = cy + 150 - (bb2[2]-bb2[0])
            draw.text((vx2, y), data["gross_pay_ytd"], fill=(0,0,0), font=fonts["value"])
            record_bbox("gross_pay_ytd", vx2, y, bb2[2]-bb2[0], bb2[3]-bb2[1])
        y += 22

    y += 8; draw_separator(draw, margin, img_width-margin, y); y += 10
    draw.text((margin, y), "DEDUCTIONS", fill=(0,0,0), font=fonts["label"])
    draw.text((cc, y), "Current", fill=(0,0,0), font=fonts["label"])
    draw.text((cy, y), "YTD", fill=(0,0,0), font=fonts["label"])
    y += 20; draw_separator(draw, margin, img_width-margin, y); y += 5
    for cid in ("federal_tax_withheld", "state_tax_withheld", "social_security_tax", "medicare_tax", "retirement_401k", "health_insurance"):
        if cid in data:
            draw.text((margin, y), labels.get(cid,cid), fill=(100,100,100), font=fonts["label"])
            bb = fonts["value"].getbbox(data[cid])
            vx = cc + 150 - (bb[2]-bb[0])
            draw.text((vx, y), data[cid], fill=(0,0,0), font=fonts["value"])
            record_bbox(cid, vx, y, bb[2]-bb[0], bb[3]-bb[1])
            y += 20

    if "total_deductions_current" in data:
        draw_separator(draw, margin, img_width-margin, y); y += 5
        draw.text((margin, y), labels.get("total_deductions_current","Total Ded"), fill=(0,0,0), font=fonts["label"])
        bb = fonts["value"].getbbox(data["total_deductions_current"])
        vx = cc + 150 - (bb[2]-bb[0])
        draw.text((vx, y), data["total_deductions_current"], fill=(0,0,0), font=fonts["value"])
        record_bbox("total_deductions_current", vx, y, bb[2]-bb[0], bb[3]-bb[1])
        if "total_deductions_ytd" in data:
            bb2 = fonts["value"].getbbox(data["total_deductions_ytd"])
            vx2 = cy + 150 - (bb2[2]-bb2[0])
            draw.text((vx2, y), data["total_deductions_ytd"], fill=(0,0,0), font=fonts["value"])
            record_bbox("total_deductions_ytd", vx2, y, bb2[2]-bb2[0], bb2[3]-bb2[1])
        y += 22

    y += 10; draw_separator(draw, margin, img_width-margin, y, color=(0,0,0)); y += 12
    if "net_pay_current" in data:
        draw.text((margin, y), labels.get("net_pay_current","Net Pay"), fill=(0,0,0), font=fonts["label"])
        bb = fonts["header"].getbbox(data["net_pay_current"])
        vx = cc + 150 - (bb[2]-bb[0])
        draw.text((vx, y), data["net_pay_current"], fill=(0,0,0), font=fonts["header"])
        record_bbox("net_pay_current", vx, y, bb[2]-bb[0], bb[3]-bb[1])
        if "net_pay_ytd" in data:
            bb2 = fonts["header"].getbbox(data["net_pay_ytd"])
            vx2 = cy + 150 - (bb2[2]-bb2[0])
            draw.text((vx2, y), data["net_pay_ytd"], fill=(0,0,0), font=fonts["header"])
            record_bbox("net_pay_ytd", vx2, y, bb2[2]-bb2[0], bb2[3]-bb2[1])


# ── Layout registry ────────────────────────────────────────────────────────
LAYOUT_TEMPLATES = {
    "corporate_standard": draw_corporate_standard,
    "left_header": draw_left_header,
    "two_column": draw_two_column,
    "compact": draw_compact,
    "check_style": draw_check_style,
    "modern_clean": draw_modern_clean,
    "government": draw_government,
    "detailed_stub": draw_detailed_stub,
    "hourly_detailed": draw_hourly_detailed,
    "retail": draw_retail,
    "reversed_sections": draw_reversed_sections,
    "side_by_side": draw_side_by_side,
    "boxed": draw_boxed,
    "minimal": draw_minimal,
    "wide_format": draw_wide_format,
}

print(f"{len(LAYOUT_TEMPLATES)} layout templates registered")

In [ ]:
# ── Renderer with Bounding Box Tracking ─────────────────────────────────────

def load_fonts(font_file, sizes=None):
    """Load a font file at multiple sizes."""
    if sizes is None:
        sizes = {"header": 22, "label": 13, "value": 13, "small": 10}
    fonts = {}
    for key, size in sizes.items():
        try:
            fonts[key] = ImageFont.truetype(str(font_file), size)
        except Exception:
            fonts[key] = ImageFont.load_default()
    return fonts


def render_paystub(template_name, template_fn, data, labels, font_file,
                   img_width=850, img_height=1100):
    """Render a paystub image and return (image, bboxes)."""
    img = Image.new("RGB", (img_width, img_height), (255, 255, 255))
    draw_obj = ImageDraw.Draw(img)
    fonts = load_fonts(font_file)

    bboxes = []
    def record_bbox(concept_id, x, y, w, h):
        bboxes.append({
            "canonical_concept_id": concept_id,
            "concept_family": CONCEPT_TO_FAMILY.get(concept_id, "unknown"),
            "surface_label": labels.get(concept_id, concept_id),
            "extracted_value": data.get(concept_id, ""),
            "display_value": data.get(concept_id, ""),
            "field_position": {"x": int(x), "y": int(y), "width": int(w), "height": int(h)},
        })

    template_fn(draw_obj, data, labels, fonts, record_bbox, img_width, img_height)
    return img, bboxes


print("Renderer ready")

In [ ]:
# ── Noise Engine (5 profiles) ───────────────────────────────────────────────

NOISE_PROFILES = {
    "clean":       {"weight": 0.40, "rotation": 0,   "blur": 0,   "jpeg_q": 95, "brightness": 1.0,  "salt_pepper": 0},
    "light_scan":  {"weight": 0.30, "rotation": 0.5, "blur": 0.5, "jpeg_q": 85, "brightness": 0.95, "salt_pepper": 0.001},
    "heavy_scan":  {"weight": 0.15, "rotation": 1.5, "blur": 1.0, "jpeg_q": 70, "brightness": 0.9,  "salt_pepper": 0.003},
    "phone_photo": {"weight": 0.10, "rotation": 2.0, "blur": 0.8, "jpeg_q": 75, "brightness": 0.88, "salt_pepper": 0.002},
    "faded":       {"weight": 0.05, "rotation": 0.3, "blur": 0.3, "jpeg_q": 80, "brightness": 0.85, "salt_pepper": 0},
}


def pick_noise_profile(rng):
    """Weighted random selection of noise profile."""
    names = list(NOISE_PROFILES.keys())
    weights = [NOISE_PROFILES[n]["weight"] for n in names]
    return rng.choices(names, weights=weights, k=1)[0]


def apply_noise(img, profile_name, rng):
    """Apply noise profile to image, return new image."""
    p = NOISE_PROFILES[profile_name]
    result = img.copy()

    # Rotation
    if p["rotation"] > 0:
        angle = rng.uniform(-p["rotation"], p["rotation"])
        result = result.rotate(angle, fillcolor=(255, 255, 255), expand=False)

    # Blur
    if p["blur"] > 0:
        result = result.filter(ImageFilter.GaussianBlur(radius=p["blur"]))

    # Brightness
    if p["brightness"] != 1.0:
        enhancer = ImageEnhance.Brightness(result)
        result = enhancer.enhance(p["brightness"])

    # Salt & pepper noise
    if p["salt_pepper"] > 0:
        arr = np.array(result)
        noise_mask = np.random.random(arr.shape[:2])
        arr[noise_mask < p["salt_pepper"] / 2] = 0       # salt (black)
        arr[noise_mask > 1 - p["salt_pepper"] / 2] = 255  # pepper (white)
        result = Image.fromarray(arr)

    # JPEG compression artifact
    if p["jpeg_q"] < 95:
        buf = io.BytesIO()
        result.save(buf, format="JPEG", quality=p["jpeg_q"])
        buf.seek(0)
        result = Image.open(buf).convert("RGB")

    return result


print(f"Noise engine ready: {list(NOISE_PROFILES.keys())}")

In [ ]:
# ── Batch Generator ─────────────────────────────────────────────────────────

def pick_labels(rng):
    """Pick random surface labels for all concepts."""
    return {cid: rng.choice(aliases) for cid, aliases in LABEL_ALIASES.items()}


def generate_batch(n=200, seed=42):
    """Generate n paystubs with round-robin layouts. Returns manifest list."""
    rng = random.Random(seed)
    gen = PaystubDataGenerator(seed=seed)
    layout_names = list(LAYOUT_TEMPLATES.keys())
    manifest = []

    for i in range(n):
        paystub_id = f"ps_{i:04d}"
        layout_name = layout_names[i % len(layout_names)]
        template_fn = LAYOUT_TEMPLATES[layout_name]

        # Generate data
        result = gen.generate()
        assert gen.validate(result["ground_truth"]), f"Validation failed for {paystub_id}"
        data = result["display"]
        ground_truth = result["ground_truth"]

        # Pick random labels and font
        labels = pick_labels(rng)
        font_file = rng.choice(FONT_FILES)

        # Pick noise
        noise_profile = pick_noise_profile(rng)

        # Render
        img, bboxes = render_paystub(layout_name, template_fn, data, labels, font_file)

        # Apply noise
        img = apply_noise(img, noise_profile, rng)

        # Save image
        img_filename = f"{paystub_id}.png"
        img.save(IMG_DIR / img_filename)

        # Build ground truth JSON
        gt_record = {
            "paystub_id": paystub_id,
            "image_file": img_filename,
            "layout_template": layout_name,
            "noise_profile": noise_profile,
            "font": font_file.name,
            "fields": bboxes,
            "ground_truth": ground_truth,
        }

        # Save individual ground truth
        with open(GT_DIR / f"{paystub_id}.json", "w") as f:
            json.dump(gt_record, f, indent=2)

        manifest.append(gt_record)

        if (i + 1) % 50 == 0:
            print(f"  Generated {i+1}/{n} paystubs...")

    print(f"\nDone! {len(manifest)} paystubs generated.")
    return manifest


print("Batch generator ready")

In [ ]:
# ── Run Generation ──────────────────────────────────────────────────────────
print("Generating 200 synthetic paystubs...")
manifest = generate_batch(n=200, seed=42)
print(f"\nImages saved to: {IMG_DIR}")
print(f"Ground truth saved to: {GT_DIR}")

In [ ]:
# ── Verification ────────────────────────────────────────────────────────────
from IPython.display import display as ipy_display

# 1. Display 5 random samples
sample_indices = random.sample(range(len(manifest)), 5)
for idx in sample_indices:
    rec = manifest[idx]
    img = Image.open(IMG_DIR / rec["image_file"])
    print(f"\n{'='*60}")
    print(f"  {rec['paystub_id']} | Layout: {rec['layout_template']} | Noise: {rec['noise_profile']}")
    print(f"  Employee: {rec['ground_truth']['employee_name']}")
    print(f"  Gross: ${float(rec['ground_truth']['gross_pay_current']):,.2f}  |  Net: ${float(rec['ground_truth']['net_pay_current']):,.2f}")
    print(f"  Fields tracked: {len(rec['fields'])}")
    # Show thumbnail
    thumb = img.copy()
    thumb.thumbnail((400, 550))
    ipy_display(thumb)

# 2. Validate all 200 ground truths have consistent math
gen_check = PaystubDataGenerator()
all_valid = True
for rec in manifest:
    if not gen_check.validate(rec["ground_truth"]):
        print(f"FAIL: {rec['paystub_id']} math inconsistency!")
        all_valid = False
print(f"\nMath validation: {'ALL 200 PASSED' if all_valid else 'FAILURES DETECTED'}")

# 3. Layout distribution
layout_counts = Counter(r["layout_template"] for r in manifest)
print(f"\nLayout distribution ({len(layout_counts)} layouts):")
for name, count in sorted(layout_counts.items()):
    status = "OK" if count >= 10 else "LOW"
    print(f"  {name:25s}: {count:3d}  [{status}]")

# 4. Surface label usage
label_usage = Counter()
for rec in manifest:
    for field in rec["fields"]:
        label_usage[field["surface_label"]] += 1
low_labels = {l: c for l, c in label_usage.items() if c < 3}
print(f"\nSurface label variants used: {len(label_usage)}")
if low_labels:
    print(f"  Labels used < 3 times: {len(low_labels)}")
else:
    print(f"  All labels used >= 3 times")

# 5. Concept coverage
concept_coverage = Counter()
for rec in manifest:
    for field in rec["fields"]:
        concept_coverage[field["canonical_concept_id"]] += 1
print(f"\nConcept coverage ({len(concept_coverage)} concepts tracked):")
for cid in ALL_CONCEPTS:
    cnt = concept_coverage.get(cid, 0)
    status = "OK" if cnt > 0 else "MISSING"
    print(f"  {cid:30s}: {cnt:4d}  [{status}]")

# 6. Noise profile distribution
noise_counts = Counter(r["noise_profile"] for r in manifest)
print(f"\nNoise profile distribution:")
for name, count in sorted(noise_counts.items()):
    print(f"  {name:15s}: {count:3d}")

# 7. Value uniqueness
value_sets = defaultdict(set)
for rec in manifest:
    for k, v in rec["ground_truth"].items():
        value_sets[k].add(v)
print(f"\nValue uniqueness (sample):")
for k in ["employee_name", "gross_pay_current", "net_pay_current", "employer_name"]:
    print(f"  {k}: {len(value_sets[k])} unique values")

In [ ]:
# ── Export Manifest ──────────────────────────────────────────────────────────
manifest_path = BASE_DIR / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved to: {manifest_path}")
print(f"Total records: {len(manifest)}")
print(f"\nFile counts:")
print(f"  Images:       {len(list(IMG_DIR.glob('*.png')))}")
print(f"  Ground truth: {len(list(GT_DIR.glob('*.json')))}")
print(f"\nPhase 3 generation complete! Ready for image-based CBA scoring.")